# Fig 6A / S13A — AD GWAS UpSet of shared credible sets (data processing)

Pairwise CS colocalizations between AD GWAS cohorts are stitched into signals (connected components within a block); each signal is marked by the cohorts it appears in.

**Outputs:**
- `figure_data/Fig6A.AD_GWAS_UpSet.rds` — SuSiE-rRSS: list(`signals`, `signals_noAPOE`, `regions`)
- `figure_data/FigS13A.AD_GWAS_UpSet_SuSiE_RSS.rds` — SuSiE-RSS: list(`signals`, `signals_noAPOE`)

In [1]:
source("../../R/utils.R")
library(igraph)

cs_coloc_df <- readRDS("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export/method_compare.cs_coloc.rds")

eb_mix <- read_gwas_tables("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export/")
eb_mix <- eb_mix %>% mutate(method = "eb_mix") %>% filter(cs_coverage_0.95 != "susie_rss_0")


Attaching package: ‘igraph’


The following objects are masked from ‘package:lubridate’:

    %--%, union


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:purrr’:

    compose, simplify


The following object is masked from ‘package:tidyr’:

    crossing


The following object is masked from ‘package:tibble’:

    as_data_frame


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union




In [2]:
## CS signals: unique node per (block_id, cs) — cs repeats across regions, so it is
## namespaced by block; colocalized CS pairs are edges; components are signals.
## Returns one row per signal with a 0/1 column per study.
signal_membership <- function(cs_coloc_df, m, exclude_blocks = character(0)) {
  df <- cs_coloc_df %>% filter(method == m) %>% filter(!block_id %in% exclude_blocks)

  node_tbl <- df %>% distinct(block_id, cs) %>%
    mutate(node = paste(block_id, cs, sep = "||"))

  key <- function(b, c) paste(b, str_trim(c), sep = "||")

  edges <- df %>%
    filter(!is.na(partners), partners != "") %>%
    select(block_id, cs, partners) %>%
    separate_rows(partners, sep = ";\\s*") %>%
    transmute(from = key(block_id, cs), to = key(block_id, partners)) %>%
    filter(from %in% node_tbl$node, to %in% node_tbl$node)   # keep edges within region/method

  g    <- graph_from_data_frame(edges, directed = FALSE,
                                vertices = data.frame(name = node_tbl$node))
  comp <- components(g)

  node_tbl %>%
    mutate(set   = comp$membership[node],
           study = str_split_fixed(cs, "\\|", 2)[, 1]) %>%
    distinct(set, study) %>%
    mutate(v = 1L) %>%
    pivot_wider(names_from = study, values_from = v, values_fill = 0L) %>%
    arrange(set) %>%
    mutate(set = paste0("set", set))
}

## REGION overlap: a region is "shared" if >= 1 study has a CS in it
region_membership <- function(df, region_col = "block_id", study_col = "study",
                              cs_index_col = "cs_coverage_0.95") {
  df %>%
    filter(.data[[cs_index_col]] != 0) %>%                         # in a credible set
    distinct(region = .data[[region_col]], study = .data[[study_col]]) %>%
    mutate(v = 1L) %>%
    pivot_wider(names_from = study, values_from = v, values_fill = 0L)
}

## Fig 6B: SuSiE-rRSS

In [3]:
upset_rrss <- list(
  signals        = signal_membership(cs_coloc_df, "ebmix"),
  signals_noAPOE = signal_membership(cs_coloc_df, "ebmix", exclude_blocks = apoe_block),
  regions        = region_membership(eb_mix)
)
save_data(upset_rrss, "Fig6A.AD_GWAS_UpSet")

## Fig S12A: SuSiE-RSS

In [4]:
upset_rss <- list(
  signals        = signal_membership(cs_coloc_df, "susie"),
  signals_noAPOE = signal_membership(cs_coloc_df, "susie", exclude_blocks = apoe_block)
)
save_data(upset_rss, "FigS13A.AD_GWAS_UpSet_SuSiE_RSS")